In [1]:
#bài 1:

# -*- coding: utf-8 -*-
"""
Bai tap 1: Tu viet KNN khong dung thu vien KNN.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Tu tinh khoang cach va du doan nhan bang KNN."""

    # Tinh khoang cach Euclid tu mau moi den cac mau hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    # Lay chi so cua k lang gieng gan nhat
    chi_so = np.argsort(khoang_cach)[:k]

    # Lay nhan cua k lang gieng
    nhan_gan = y_hoc[chi_so]

    # Dem so lan xuat hien cua tung nhan va chon nhan pho bien nhat
    cac_nhan, so_luong = np.unique(nhan_gan, return_counts=True)
    return cac_nhan[np.argmax(so_luong)]


# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/hai_vang_trang.csv")

X = df[["x1", "x2"]].to_numpy(dtype=float)
y = df["nhan"].to_numpy()

# Chia tap huan luyen va tap kiem tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Du doan bang KNN tu viet voi k = 13
k = 13

y_du_doan_tu_viet = np.array([
    knn_du_doan(X_tr, y_tr, x, k)
    for x in X_te
])

acc_tu_viet = accuracy_score(y_te, y_du_doan_tu_viet)

# So sanh voi KNeighborsClassifier
mo_hinh = KNeighborsClassifier(n_neighbors=k)
mo_hinh.fit(X_tr, y_tr)

y_du_doan_thu_vien = mo_hinh.predict(X_te)
acc_thu_vien = accuracy_score(y_te, y_du_doan_thu_vien)

# In ket qua
print("Ket qua KNN voi k = 13")
print(f"Accuracy KNN tu viet: {acc_tu_viet:.4f}")
print(f"Accuracy KNeighborsClassifier: {acc_thu_vien:.4f}")

print(
    "Hai ket qua giong nhau:",
    np.isclose(acc_tu_viet, acc_thu_vien)
)


Ket qua KNN voi k = 13
Accuracy KNN tu viet: 0.9200
Accuracy KNeighborsClassifier: 0.9200
Hai ket qua giong nhau: True


In [2]:
# bài 2:

# -*- coding: utf-8 -*-
"""
Bai tap 2: So sanh khoang cach Manhattan va Euclid.
"""

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia tap huan luyen va tap kiem tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Thu cac gia tri k le tu 1 den 51
cac_k = range(1, 52, 2)

# So sanh Euclid va Manhattan
for metric in ["euclidean", "manhattan"]:
    diem = {}

    for k in cac_k:
        mo_hinh = KNeighborsClassifier(
            n_neighbors=k,
            metric=metric
        )

        diem[k] = cross_val_score(
            mo_hinh,
            X_tr,
            y_tr,
            cv=5
        ).mean()

    # Chon k co diem kiem dinh cheo cao nhat
    k_tot = max(diem, key=diem.get)

    # Danh gia tren tap kiem tra
    mo_hinh = KNeighborsClassifier(
        n_neighbors=k_tot,
        metric=metric
    )
    mo_hinh.fit(X_tr, y_tr)

    y_du_doan = mo_hinh.predict(X_te)
    acc = accuracy_score(y_te, y_du_doan)

    print(f"Khoang cach: {metric}")
    print(f"k tot nhat: {k_tot}")
    print(f"Accuracy tap kiem tra: {acc:.4f}")
    print()


Khoang cach: euclidean
k tot nhat: 13
Accuracy tap kiem tra: 0.9200

Khoang cach: manhattan
k tot nhat: 27
Accuracy tap kiem tra: 0.9067



In [3]:
#bài 3

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/sinh_vien_mo_rong.csv")

# Doi chi_tieu tu nghin dong sang trieu dong
df["chi_tieu"] = df["chi_tieu"] / 1000

y = df["qua_mon"]
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

# Chia du lieu giong muc 3
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# KNN k = 7, khong chuan hoa
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_tr, y_tr)

y_du_doan = knn.predict(X_te)
acc_khong_chuan = accuracy_score(y_te, y_du_doan)

print("KNN k = 7, ba cot khong chuan hoa")
print(f"Accuracy: {acc_khong_chuan:.4f}")

# KNN k = 7, co chuan hoa
knn_chuan = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=7)
)
knn_chuan.fit(X_tr, y_tr)

y_du_doan_chuan = knn_chuan.predict(X_te)
acc_chuan = accuracy_score(y_te, y_du_doan_chuan)

print()
print("KNN k = 7, ba cot co chuan hoa")
print(f"Accuracy: {acc_chuan:.4f}")
print("Accuracy muc 3 truoc khi doi don vi: 0.5333")


KNN k = 7, ba cot khong chuan hoa
Accuracy: 0.8667

KNN k = 7, ba cot co chuan hoa
Accuracy: 0.8667
Accuracy muc 3 truoc khi doi don vi: 0.5333


In [5]:
# bài 4:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia tap huan luyen va tap kiem tra giong muc 4
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# So sanh bo phieu deu va bo phieu co trong so
print("So sanh KNN uniform va distance")
print(" k    uniform    distance")

for k in [1, 13, 101]:
    # Bo phieu deu
    knn_deu = KNeighborsClassifier(
        n_neighbors=k,
        weights="uniform"
    )
    knn_deu.fit(X_tr, y_tr)
    acc_deu = accuracy_score(y_te, knn_deu.predict(X_te))

    # Bo phieu theo khoang cach
    knn_trong_so = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance"
    )
    knn_trong_so.fit(X_tr, y_tr)
    acc_trong_so = accuracy_score(
        y_te,
        knn_trong_so.predict(X_te)
    )

    print(f"{k:3d}   {acc_deu:.4f}     {acc_trong_so:.4f}")


So sanh KNN uniform va distance
 k    uniform    distance
  1   0.8000     0.8000
 13   0.9200     0.9333
101   0.8533     0.9067


In [ ]:
#bài tập 5:

# -*- coding: utf-8 -*-
"""
Bai tap 5: Chon k cho bai toan hoi quy.
"""

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Doc du lieu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/gia_nha.csv")

X = df[["dien_tich"]]
y = df["gia"]

# Chia tap huan luyen va tap kiem tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Chon k tot nhat bang kiem dinh cheo 5 phan
diem = {}

for k in range(1, 21):
    mo_hinh = KNeighborsRegressor(n_neighbors=k)

    scores = cross_val_score(
        mo_hinh,
        X_tr,
        y_tr,
        cv=5,
        scoring="neg_mean_absolute_error"
    )

    # Diem tra ve mang dau am, doi lai thanh MAE duong
    diem[k] = -scores.mean()

k_tot = min(diem, key=diem.get)

print("MAE trung binh kiem dinh cheo theo k")
for k, mae in diem.items():
    print(f"k = {k:2d}: MAE = {mae:.4f}")

print()
print(f"k tot nhat = {k_tot}")
print(f"MAE kiem dinh cheo tot nhat = {diem[k_tot]:.4f}")

# Huan luyen KNN voi k tot nhat
knn = KNeighborsRegressor(n_neighbors=k_tot)
knn.fit(X_tr, y_tr)

mae_knn = mean_absolute_error(y_te, knn.predict(X_te))

# Huan luyen duong hoi quy tuyen tinh de so sanh
duong = LinearRegression()
duong.fit(X_tr, y_tr)

mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print()
print("So sanh MAE tren tap kiem tra (ty dong)")
print(f"KNN, k = {k_tot}: {mae_knn:.4f}")
print(f"Duong hoi quy tuyen tinh: {mae_duong:.4f}")

if mae_knn < mae_duong:
    print("KNN co sai so thap hon duong hoi quy tuyen tinh.")
elif mae_knn > mae_duong:
    print("KNN co sai so cao hon duong hoi quy tuyen tinh.")
else:
    print("Hai mo hinh co MAE bang nhau.")
